In [5]:
import pandas as pd
import random
import numpy as np
import cv2
import json

In [6]:
df = pd.read_csv('data/identity_CelebA.txt', sep=' ', header=None, names=['image_id', 'identity_id'])
identity = {}
for i in range(len(df)):
    if df['identity_id'][i]-1 not in identity: identity[df['identity_id'][i]-1] = []
    identity[df['identity_id'][i]-1].append(df['image_id'][i])
key = list(identity.keys())
random.seed(42)
random.shuffle(key)
ctr = 0
identity1 = {}
for k in key:
    identity1[ctr] = identity[k]
    ctr += 1
cur = valnum = testnum = 0
arr = []
for i in range(len(identity1)):
    cur += len(identity1[i])
    for j in range(len(identity1[i])): arr.append(i)
    if cur >= 0.8*len(df) and valnum == 0: valnum = cur
    if cur >= 0.9*len(df) and testnum == 0: testnum = cur

In [9]:
def get_data(n, size, split):
    random.seed(42)
    start = {'train': 0, 'val': valnum, 'test': testnum}[split]
    end = {'train': valnum-1, 'val': testnum-1, 'test': len(arr)-1}[split]
    match = set()
    while len(match) < size // 2:
        i = arr[random.randint(start, end)]
        if len(identity1[i]) < n: continue
        s = set()
        while len(s) < n:
            j = random.randint(0, len(identity1[i])-1)
            s.add(identity1[i][j])
        l = sorted(s)
        random.shuffle(l)
        l = sorted(l[:-1]) + [l[-1]]
        match.add((tuple(l), i))
    mismatch = set()
    while len(mismatch) < size // 2:
        i = arr[random.randint(start, end)]
        if len(identity1[i]) < n-1: continue
        s = set()
        while len(s) < n-1:
            j = random.randint(0, len(identity1[i])-1)
            s.add(identity1[i][j])
        l = sorted(s)
        j, m = i, 0
        while j == i:
            j = arr[random.randint(start, end)]
            m = random.randint(0, len(identity1[j])-1)
        l.append(identity1[j][m])
        mismatch.add((tuple(l), i))
    if size % 2 == 1: size -= 1
    pairs = sorted(match.union(mismatch))
    path = 'data/img_align_celeba/'
    inputs_memmap = np.memmap(f'{split}_data1/inputs_{n}.dat', dtype=np.uint8, mode='w+', shape=(size, n, 128, 128, 3))
    targets_memmap = np.memmap(f'{split}_data1/targets_{n}.dat', dtype=np.uint8, mode='w+', shape=(size,))
    identity2 = []
    for i, j in enumerate(pairs):
        targets_memmap[i] = int(j in match)
        for k, img in enumerate(j[0]):
            img1 = cv2.resize(cv2.imread(path + img), (128, 128), interpolation=cv2.INTER_AREA)
            inputs_memmap[i, k] = img1
        if i % 10000 == 9999 or i == size - 1:
            inputs_memmap.flush()
            targets_memmap.flush()
        identity2.append(j[1])
    return identity2

In [10]:
get_data(2, 200000, 'train')
get_data(2, 20000, 'val')
with open('identity/test_2.json', 'w') as file:
    json.dump(get_data(2, 20000, 'test'), file)

In [11]:
get_data(3, 200000, 'train')
get_data(3, 20000, 'val')
with open('identity/test_3.json', 'w') as file:
    json.dump(get_data(3, 20000, 'test'), file)

In [12]:
get_data(4, 200000, 'train')
get_data(4, 20000, 'val')
with open('identity/test_4.json', 'w') as file:
    json.dump(get_data(4, 20000, 'test'), file)

In [13]:
get_data(5, 200000, 'train')
get_data(5, 20000, 'val')
with open('identity/test_5.json', 'w') as file:
    json.dump(get_data(5, 20000, 'test'), file)

In [14]:
get_data(6, 200000, 'train')
get_data(6, 20000, 'val')
with open('identity/test_6.json', 'w') as file:
    json.dump(get_data(6, 20000, 'test'), file)